In [ ]:
import torch
import random
import re
from collections import defaultdict
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

TARGET_SIZES = {
    "train": 20000,
    "validation": 2000,
    "test": 2000
}

T5_MODEL_NAME = "google/flan-t5-base"
SENTIMENT_MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

def length_filter(example):
    """Filter reviews between 50 and 500 characters."""
    text = example["review_body"]
    return 50 <= len(text) <= 500

def stratified_sample(ds, total_size):
    label_to_indices = defaultdict(list)
    for idx, example in enumerate(ds):
        label_to_indices[example["stars"]].append(idx)

    num_classes = len(label_to_indices)
    per_class = total_size // num_classes
    remainder = total_size % num_classes

    if remainder != 0:
        print(f"Warning: total_size {total_size} is not evenly divisible by {num_classes} classes. "
              f"{remainder} rows will be dropped. Actual sample size: {total_size - remainder}")

    selected_indices = []
    for label, indices in sorted(label_to_indices.items()):
        random.shuffle(indices)
        available = indices[:per_class]
        if len(available) < per_class:
            print(f"Warning: star class '{label}' only has {len(available)} rows, "
                  f"wanted {per_class}. Using all available.")
        selected_indices.extend(available)

    random.shuffle(selected_indices)
    print(f"Sampled {len(selected_indices)} rows across {num_classes} classes "
          f"({per_class} per class). Target was {total_size}.")
    return ds.select(selected_indices)


def get_summarizer_logic(tokenizer, model, device):

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def generate_summary(batch):
        prompts = [
            f"Summarize this product review in one factual sentence. Do not repeat the review text word-for-word. "
            f"Focus on the main praise or complaint.\n\n"
            f"Review: {text}\n\nSummary:"
            for text in batch["review_body"]
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=80,
                do_sample=False,
                no_repeat_ngram_size=3,
                repetition_penalty=1.5
            )

        summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        return {"summary": summaries}

    return generate_summary


def get_bias_logic(sentiment_model):
    strong_positive = {
        "perfect", "amazing", "outstanding", "flawless", "excellent",
        "love", "fantastic", "best", "highly recommend", "life changing",
        "must buy", "great", "brilliant"
    }
    strong_negative = {
        "worst", "terrible", "awful", "horrible", "disgusting", "useless",
        "waste", "garbage", "do not buy", "never again", "absolute garbage",
        "total waste", "dreadful", "appalling"
    }
    toxic_words = {
        "shit", "fuck", "fucking", "crap", "rip off", "ripoff",
        "bullshit", "scam", "fraud"
    }
    strong_rejection = [
        "do not buy", "never buy", "waste of money",
        "not worth it", "avoid", "return immediately"
    ]
    issue_words = [
        "leak", "broken", "small", "minor",
        "slightly", "sometimes", "occasionally"
    ]

    def detect_bias(batch):
        texts = batch["review_body"]
        stars = batch["stars"]
        bias_labels = []

        sentiments = sentiment_model(texts)

        for text, star, sent in zip(texts, stars, sentiments):
            text_lower = text.lower()
            bias_score = 0.0

            label_str = sent["label"]
            star_match = re.search(r"\d", label_str)
            pred_stars = int(star_match.group()) if star_match else 3
            confidence = sent["score"]
            diff = abs(pred_stars - star)

            if diff >= 3:   bias_score += 4.0 * confidence
            elif diff == 2: bias_score += 2.5 * confidence
            elif diff == 1: bias_score += 1.0 * confidence

            pos_hits = sum(1 for w in strong_positive if w in text_lower)
            neg_hits = sum(1 for w in strong_negative if w in text_lower)

            if pos_hits >= 2 and star <= 2:
                bias_score += 1.5 + (pos_hits - 2) * 0.5
            elif pos_hits >= 1 and star == 1:
                bias_score += 1.0

            if neg_hits >= 2 and star >= 4:
                bias_score += 1.5 + (neg_hits - 2) * 0.5
            elif neg_hits >= 1 and star == 5:
                bias_score += 1.0

            toxic_hits = sum(1 for w in toxic_words if w in text_lower)
            if toxic_hits > 0:
                bias_score += (2.0 * toxic_hits) if star >= 4 else (0.5 * toxic_hits)

            if pos_hits >= 2 and neg_hits >= 2:   bias_score += 1.5
            elif pos_hits >= 1 and neg_hits >= 1: bias_score += 0.5

            rej_hits = sum(1 for w in strong_rejection if w in text_lower)
            issue_hits = sum(1 for w in issue_words if w in text_lower)
            if rej_hits > 0 and issue_hits <= 2 and star <= 2:
                bias_score += 1.5

            bias_labels.append(1 if bias_score >= 2.0 else 0)

        return {"bias_label": bias_labels}

    return detect_bias


def flush_gpu_cache(label: str):
    """Clears PyTorch's CUDA memory allocator cache and prints allocator stats."""
    if torch.cuda.is_available():
        allocated_before = torch.cuda.memory_allocated() / 1024 ** 2
        torch.cuda.empty_cache()
        allocated_after = torch.cuda.memory_allocated() / 1024 ** 2
        print(f"[{label}] GPU cache flushed. "
              f"Allocated: {allocated_before:.1f} MB -> {allocated_after:.1f} MB")
    else:
        print(f"[{label}] CPU mode — no GPU cache to flush.")


def main():
    random.seed(42)

    print("Loading dataset...")
    dataset = load_dataset("goosmanlei/amazon_reviews_multi", "en")

    print("Filtering by review length (50–500 chars)...")
    filtered_ds = dataset.filter(length_filter)

    print("Stratified sampling...")
    sampled_dataset = DatasetDict()
    for split in ["train", "validation", "test"]:
        sampled_dataset[split] = stratified_sample(
            filtered_ds[split], TARGET_SIZES[split]
        )

    print(f"\nLoading {T5_MODEL_NAME}...")
    t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_NAME)
    t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME).to(device)
    t5_model.eval()

    print(f"Loading {SENTIMENT_MODEL_NAME}...")
    sentiment_pipe = pipeline(
        "sentiment-analysis",
        model=SENTIMENT_MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
        truncation=True,
        max_length=512
    )

    flush_gpu_cache("after model load")

    final_dataset = DatasetDict()

    for split in ["train", "validation", "test"]:
        print(f"\n{'='*50}")
        print(f"Processing split: {split}  ({len(sampled_dataset[split])} rows)")
        print(f"{'='*50}")

        print("Pass 1 — Generating summaries...")
        ds_with_summaries = sampled_dataset[split].map(
            get_summarizer_logic(t5_tokenizer, t5_model, device),
            batched=True,
            batch_size=16,
            desc=f"Summarising {split}"
        )
        flush_gpu_cache(f"after summarisation [{split}]")

        print("Pass 2 — Detecting bias...")
        ds_final = ds_with_summaries.map(
            get_bias_logic(sentiment_pipe),
            batched=True,
            batch_size=8,
            desc=f"Bias detection {split}"
        )
        flush_gpu_cache(f"after bias detection [{split}]")

        final_dataset[split] = ds_final

        save_path = f"/workspace/augmentated_amazon_reviews/{split}"
        os.makedirs(save_path, exist_ok=True)
        ds_final.save_to_disk(save_path)
        print(f"Saved {split} split to {save_path}  "
              f"({len(ds_final)} rows, columns: {ds_final.column_names})")

    print("\n" + "="*50)
    print("All splits processed and saved.")
    for split in ["train", "validation", "test"]:
        ds = final_dataset[split]
        bias_rate = sum(ds["bias_label"]) / len(ds) * 100
        print(f"  {split:10s}: {len(ds):>6} rows | "
              f"bias_label=1: {bias_rate:.1f}%")

    example = final_dataset["train"][0]
    print("\nExample entry:")
    print(f"  Review : {example['review_body'][:120]}...")
    print(f"  Summary: {example['summary']}")
    print(f"  Stars  : {example['stars']}  |  Bias label: {example['bias_label']}")


    final_dataset.save_to_disk("augmentated_amazon_reviews_full")


if __name__ == "__main__":
    main()

c:\Users\manish\Downloads\Karanveer\GenAI-Project\augment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading dataset...
Filtering by review length (50–500 chars)...
Stratified sampling...
Sampled 20000 rows across 5 classes (4000 per class). Target was 20000.
Sampled 2000 rows across 5 classes (400 per class). Target was 2000.
Sampled 2000 rows across 5 classes (400 per class). Target was 2000.

Loading google/flan-t5-base...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3592.02it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading nlptown/bert-base-multilingual-uncased-sentiment...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5466.65it/s]


...

Bias detection test: 100%|██████████| 2000/2000 [00:22<00:00, 88.26 examples/s]


[after bias detection [test]] GPU cache flushed. Allocated: 1594.3 MB -> 1594.3 MB


Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 210198.66 examples/s]


Saved test split to /workspace/augmentated_amazon_reviews/test  (2000 rows, columns: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category', 'summary', 'bias_label'])

All splits processed and saved.
  train     :  20000 rows | bias_label=1: 1.7%
  validation:   2000 rows | bias_label=1: 1.7%
  test      :   2000 rows | bias_label=1: 2.2%

Example entry:
  Review : I am currently using this for iPad, iPhone mostly for charging with an adaptor. Pros: 1. The cable is nylon-braided so i...
  Summary: Great cable.
  Stars  : 4  |  Bias label: 0


Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 209940.89 examples/s]


In [ ]:
import os
import random
import pandas as pd
from datasets import load_from_disk

DATASET_PATH = "preprocessed_amazon_reviews_full/train"  
OUTPUT_FILE = "manual_audit_results.csv"
NUM_SAMPLES_TO_CHECK = 20  

def run_audit():
    if not os.path.exists(DATASET_PATH):
        print(f"Error: Path {DATASET_PATH} not found.")
        return

    ds = load_from_disk(DATASET_PATH)

    indices = random.sample(range(len(ds)), min(NUM_SAMPLES_TO_CHECK, len(ds)))
    
    audit_data = []

    print(f"--- Starting Manual Audit ({NUM_SAMPLES_TO_CHECK} samples) ---")
    print("Instructions: Rate the quality. Enter 'y' for Yes/Good, 'n' for No/Bad.\n")

    for i, idx in enumerate(indices):
        item = ds[idx]
        
        print(f"\n[{i+1}/{NUM_SAMPLES_TO_CHECK}] --- Index: {idx}")
        print(f"STARS:   {item['stars']}")
        print(f"REVIEW:  {item['review_body']}")
        print("-" * 30)
        print(f"SUMMARY: {item['summary']}")
        print(f"BIAS:    {item['bias_label']} (1=Biased, 0=Unbiased)")
        print("-" * 30)

        sum_grade = input("Is the summary accurate/concise? (y/n): ").lower().strip()
        bias_grade = input("Is the bias label logically correct? (y/n): ").lower().strip()
        
        audit_data.append({
            "index": idx,
            "stars": item['stars'],
            "review": item['review_body'],
            "gen_summary": item['summary'],
            "gen_bias": item['bias_label'],
            "summary_correct": 1 if sum_grade == 'y' else 0,
            "bias_correct": 1 if bias_grade == 'y' else 0
        })


    df = pd.DataFrame(audit_data)

    if os.path.exists(OUTPUT_FILE):
        existing_df = pd.read_csv(OUTPUT_FILE)
        df = pd.concat([existing_df, df], ignore_index=True)
    
    df.to_csv(OUTPUT_FILE, index=False)

    sum_acc = (df['summary_correct'].sum() / len(df)) * 100
    bias_acc = (df['bias_correct'].sum() / len(df)) * 100

    print("\n" + "="*40)
    print("AUDIT COMPLETE")
    print(f"Results saved to: {OUTPUT_FILE}")
    print(f"Total samples audited so far: {len(df)}")
    print(f"Current Summary Accuracy: {sum_acc:.1f}%")
    print(f"Current Bias Label Accuracy: {bias_acc:.1f}%")
    print("="*40)

if __name__ == "__main__":
    run_audit()

--- Starting Manual Audit (20 samples) ---
Instructions: Rate the quality. Enter 'y' for Yes/Good, 'n' for No/Bad.


[1/20] --- Index: 1927
STARS:   1
REVIEW:  I won't recommend this product is not working after you leave the house. Is a trash product.
------------------------------
SUMMARY: Don't waste your time or money on this product.
BIAS:    0 (1=Biased, 0=Unbiased)
------------------------------

[2/20] --- Index: 11448
STARS:   4
REVIEW:  This worked out really well for me. I was consistently getting 90 pumps or water out of the 5 ml size bottle. That is not a typo. Ninety.
------------------------------
SUMMARY: This worked out really well for me. I was consistently getting 90 pumps or water out of the 5 ml size bottle. That is not a typo. Ninety.
BIAS:    0 (1=Biased, 0=Unbiased)
------------------------------

[3/20] --- Index: 14981
STARS:   3
REVIEW:  Cousin Miguel had scabies. Black pads cleaned his skin OK.
------------------------------
SUMMARY: The product is good.
BIA